In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/long-running-durable/long-running-agentic/long-running-agents-gcp/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 04 · ADK 2 Workflow — practice
Reference: `notebooks/solutions/ex4_adk_workflow.py`.

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))          # repo root when run from notebooks/
sys.path[:0] = [os.path.join(ROOT, "src"), os.path.join(ROOT, "notebooks")]

def show_journal(run):
    print(f"run {run.run_id}  status={run.status.value}  version={run.version}  steps={run.usage.steps}  tokens={run.usage.tokens}  cost=${run.usage.cost_usd:.4f}")
    for s in run.journal:
        out = json.dumps(s.output, default=str)[:70] if s.output is not None else (s.error or "")
        print(f"  [{s.index}] {s.kind.value:<6} {s.status.value:<7} {s.name:<18} key={s.idempotency_key or '-':<20} {out}")

In [ ]:
from google.adk.events.request_input import RequestInput
from google.adk.workflow import START, Workflow, node
from lragents.adk.nightly_workflow import ASK_BUDGET, WAKE, Venue
from lragents.practice_checks import check_adk_workflow

## Exercise — build the graph
Complete the nodes. Requirements the checker enforces:
* `agree_budget` interrupts with `interrupt_id=ASK_BUDGET` until `ctx.resume_inputs[ASK_BUDGET]["budget"]` is present; it must **re-run on resume**.
* `queue_up` calls `venue.join_queue(event_id, idempotency_key=...)` and must **never re-run on resume** (one ticket across all wake-ups).
* `check_front` interrupts with `interrupt_id=WAKE` while `venue.position(ticket) > 0`; when at the front it sets `ctx.route` to `"ready"` if `venue.seats_left(event_id) >= 2` else `"sold_out"`; it must re-run on resume.
* `buy` calls `venue.purchase(event_id, 2, idempotency_key=...)` and stores the order in `ctx.state["order"]`.
* Wire the edges: `START → agree_budget → plan → queue_up → check_front → {"ready": buy, "sold_out": abandon}`.

In [ ]:
def build_workflow(venue):
    @node(rerun_on_resume=True)
    def agree_budget(ctx):
        said = ((ctx.resume_inputs or {}).get(ASK_BUDGET) or {}).get("budget")
        # TODO: if not said → return RequestInput(interrupt_id=ASK_BUDGET, message="...")
        # TODO: else store float(said) in ctx.state["budget_per_seat"] and return it
        raise NotImplementedError

    @node
    def plan(ctx):
        ctx.state["event_id"] = ctx.state["events"][0]["id"]
        return {"event_id": ctx.state["event_id"]}

    @node(rerun_on_resume=None)   # TODO: True or False? think: is this a side effect or a re-check?
    def queue_up(ctx):
        # TODO: t = venue.join_queue(ctx.state["event_id"], idempotency_key=f"{ctx.session.id}:{ctx.state['event_id']}")
        # TODO: ctx.state["ticket"] = t["ticket"]; return t
        raise NotImplementedError

    @node(rerun_on_resume=None)   # TODO: True or False?
    def check_front(ctx):
        pos = venue.position(ctx.state["ticket"])
        # TODO: interrupt while pos > 0 (interrupt_id=WAKE)
        # TODO: else set ctx.route based on venue.seats_left(...) and return a dict
        raise NotImplementedError

    @node
    def buy(ctx):
        # TODO
        raise NotImplementedError

    @node
    def abandon(ctx):
        ctx.state["order"] = None
        return {"abandoned": True}

    return Workflow(name="practice", edges=[  # TODO: the chain with the routing map
    ])

In [ ]:
print(check_adk_workflow(build_workflow))

## Design questions
1. Which of these nodes could be an `LlmAgent` instead of a function, and which must never be? Why?
2. Where does the ticket live so that it survives a Cloud Run restart? What would break if it were in a `temp:` state key?
3. Sketch the Cloud Scheduler → Pub/Sub → `/wake` payload. What identifies the paused run?

In [ ]:
answers = '''
1.
2.
3.
'''

In [ ]:
import inspect
from solutions import ex4_adk_workflow as ref
print(inspect.getsource(ref.build_workflow))